In [29]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings, GoogleGenerativeAI
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled, NoTranscriptFound, VideoUnavailable
from langchain_community.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import PromptTemplate
from dotenv import load_dotenv
load_dotenv()

True

Indexing

In [110]:
video_id = "RRVYpIET_RU"
try:
    ytt_api = YouTubeTranscriptApi()
    fetched = ytt_api.fetch(video_id, languages=["en"])
    raw_transcript = fetched.to_raw_data()
    transcript = " ".join(entry["text"] for entry in raw_transcript)
    # print(raw_transcript)
    # print("--------------------------")
    # print(type(raw_transcript))

except TranscriptsDisabled:
    print("No captions available for this video.")
except NoTranscriptFound:
    print("No transcript found in the requested language.")
except VideoUnavailable:
    print("The video is unavailable.")
except Exception as e:
    print("An unexpected error occurred:", str(e))

In [111]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)
chunks = text_splitter.create_documents([transcript])

In [112]:
embedding_model = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004" )
vector_store = FAISS.from_documents(
    chunks, embedding_model
)
print(vector_store.index_to_docstore_id)

{0: 'c157a45e-ea40-4f8a-a216-00d604603387', 1: '97170bad-a3ba-4b21-87bf-c3e7e83c47c0', 2: '29591020-c3f2-4c3f-b4c5-0b1e1f3e108e', 3: 'ec0bc806-b723-48df-ae04-330df8b03166', 4: '38e2cfac-ee33-489b-8934-044771485ca7', 5: '766580c9-bd4f-4686-afb6-8a01017b1393', 6: '4ac4f306-680f-4355-a8e5-ae65c6ed2835', 7: 'da78d520-47b0-476c-88cc-9ae2e89ffc58', 8: '6ed1dac5-6921-4bc9-83bb-afd5e2c00049', 9: '6680a505-92cf-4820-bbe5-9c6c884fa369', 10: 'a5e34dd7-e41c-4678-bd5e-a792bb979fad', 11: '22a5574c-f455-408f-820f-6d80cef7e6bd', 12: '2eaa6b1f-11c0-471f-ab40-54835e1d18ab', 13: '5733defc-f91a-4f40-b496-24d02bec352a', 14: '23f6b144-0c55-459c-a5a0-10a66cb38f9e', 15: '6bf71b2b-8061-40e0-ad0e-3ebe8978d0c1', 16: '4f9dd42f-63a2-4993-b998-7dfb76808f5e', 17: '3dcff7f7-7f2e-476c-81dc-ed458dccf5b8', 18: '908be2ef-20c4-486e-9c7f-645f0f27216c', 19: 'c5417ba4-5556-4cc2-a1c5-f49e2063952f', 20: 'f988b86b-bd70-4b87-b42a-351790a3d831', 21: '019960b9-b701-42a9-87b7-a68072722498', 22: '22b63326-1149-4633-b388-f1c6cf47bdb3

In [113]:
vector_store.get_by_ids(["a761f1f4-b2e7-486a-b5ab-1ce7ec1bf360"])

[]

Retrieval

In [114]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [115]:
retriever.invoke('What is deepmind')

[Document(id='766580c9-bd4f-4686-afb6-8a01017b1393', metadata={}, page_content="now why coding ninjas because engage your courses well structured so learning and also the courses are very well curated because these courses are prepared by people who have been at iit's as well as stanford and by the people who have been at amazon facebook and google now if you don't believe it you can check out the facebook as well as the google rating of coding ninjas the best thing about them that i find personally is the doubt resolution time like the average doubt resolution time in the last one year has been 10 minutes like if you're raising it out it gets solved within 10 minutes so that's the amazing that's the most amazing thing that they do provide so if you are looking out to buy any of the courses you can check out the link in the description you can easily get an additional 20 discount on whatever price that has been going around so guys make sure you check out coding ninjas the link will be

Augumentation

In [116]:
llm = GoogleGenerativeAI(model="gemini-2.5-pro", temperature=0.2)

In [117]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [126]:
question          = "Explain vectors with the implementation and syntax?"
retrieved_docs    = retriever.invoke(question)

In [127]:
retrieved_docs

[Document(id='5733defc-f91a-4f40-b496-24d02bec352a', metadata={}, page_content="you remember well enough this gives access to five possible indexes the first one being zero the second one being one two three four now but afterwards if you want to modify it like if i want that i want to enter one more element i cannot modify the size of this array because this array has been declared of size 5 and i cannot modify the size because arrays are constant in size so this is where something like vector comes in vector is a container which is dynamic in nature like you can always increase the size of the vector whenever you wish to so if there is a requirement where you do not know the size of a particular data structure that will be required that's when you think of a vector and that is the best place to use vector so vector is a container only okay which stores elements in a similar fashion as the array does okay now in order to declare vector it's very simple you just give the vector name th

In [128]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"you remember well enough this gives access to five possible indexes the first one being zero the second one being one two three four now but afterwards if you want to modify it like if i want that i want to enter one more element i cannot modify the size of this array because this array has been declared of size 5 and i cannot modify the size because arrays are constant in size so this is where something like vector comes in vector is a container which is dynamic in nature like you can always increase the size of the vector whenever you wish to so if there is a requirement where you do not know the size of a particular data structure that will be required that's when you think of a vector and that is the best place to use vector so vector is a container only okay which stores elements in a similar fashion as the array does okay now in order to declare vector it's very simple you just give the vector name then you give the data type it can be integer double cache string anything and\n\

In [129]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [130]:
final_prompt

StringPromptValue(text="\n      You are a helpful assistant.\n      Answer ONLY from the provided transcript context.\n      If the context is insufficient, just say you don't know.\n\n      you remember well enough this gives access to five possible indexes the first one being zero the second one being one two three four now but afterwards if you want to modify it like if i want that i want to enter one more element i cannot modify the size of this array because this array has been declared of size 5 and i cannot modify the size because arrays are constant in size so this is where something like vector comes in vector is a container which is dynamic in nature like you can always increase the size of the vector whenever you wish to so if there is a requirement where you do not know the size of a particular data structure that will be required that's when you think of a vector and that is the best place to use vector so vector is a container only okay which stores elements in a similar 

In [131]:
answer = llm.invoke(final_prompt)
answer

'Based on the transcript provided:\n\nA vector is a container that is dynamic in nature, meaning you can always increase its size. Unlike an array, which is constant in size, a vector is useful when you do not know the required size of the data structure beforehand. It stores elements in a similar fashion to an array.\n\n**Declaration and Syntax:**\n\n*   **To declare an empty vector:** You provide the vector name, the data type (e.g., integer, double, string), and then a name for the vector. This creates an empty container.\n    *   Example description: `vector <data_type> v;`\n\n*   **To declare a vector with a predefined size and initial values:** You can declare a vector of a certain size with all elements initialized to a specific value.\n    *   Example description: A declaration that results in "a container of five instances of 20."\n\n**Implementation/Functions:**\n\n*   **`push_back()`**: This function adds an element to the back of the vector. If the vector is empty, it will 